# Taller II — Integración de fuentes heterogéneas con PySpark

**Curso:** Gestión de datos y tecnologías Big Data (MCDI502) · Semana 3 · Sumativa 3
**Grupo:** 1
**Integrantes:** Yerko Gallardo, Sebastián Rojas, José Ignacio Rodríguez

**Objetivo:** capturar 100 registros de cuatro fuentes heterogéneas (dos API públicas y dos bases de datos de sensores ESP32), limpiarlos y transformarlos con Spark, integrarlos en un único DataFrame y responder consultas con Spark SQL.

| Punto de la consigna | Sección |
|---|---|
| 1. Configuración inicial del entorno | 1 |
| 2. Ingesta de datos y creación de DataFrames (ID3.1) | 2.1 OpenMeteo · 2.2 AQICN · 2.3 MongoDB · 2.4 MySQL |
| 3. Preprocesamiento y transformación de datos (RA4, ID4.2) | 3.1 `altura_metros` · 3.2 `condicion_clima` |
| 4. Integración de datos (RA3) | 4 |
| 5. Consultas y extracción de datos (ID3.1, ID4.1) | 5 (C1–C7) |
| Fuentes y atribuciones | 6 |

**Ejecución:** *Reiniciar y ejecutar todo*, sin intervención manual. El notebook funciona en Google Colab y en local (VS Code con el entorno uv del repositorio). Los datos de acceso a las fuentes (URI de MongoDB y token de AQICN) están en la sección 1.3.

## 1. Configuración inicial del entorno

| Componente | Versión | Motivo |
|---|---|---|
| Python | 3.14 en local (uv) · la de Colab | Fijada en `.python-version` |
| PySpark (incluye Apache Spark) | 4.3.0 | Fijada en `uv.lock`; se instala si falta o difiere |
| Java (JDK) | 17 o superior | Spark 4 exige Java 17 |
| PyMongo · requests | según `uv.lock` | Conector de MongoDB y cliente HTTP para las API y el endpoint PHP de MySQL |

**Conectores y versiones:**

- **MongoDB** se lee con **PyMongo**. El conector de Spark para MongoDB 3.x está compilado para Spark 3 (Scala 2.12) y no es compatible con Spark 4 (Scala 2.13). Además, PyMongo permite pedir al servidor solo los 100 documentos más recientes, en lugar de cargar la colección completa.
- **MySQL** se consulta mediante el endpoint REST entregado (`obtener_datos.php`), que devuelve la tabla como JSON. Por eso no se necesita `mysql-connector-python` ni un controlador JDBC.
- **Java 17:** Spark 4 requiere Java 17 o superior, por lo que no se usa Java 8.

### 1.1 Instalación de Spark, Java y librerías, y variables de entorno

- **Spark:** se instala como el paquete `pyspark==4.3.0`. Si ya está en esa versión (entorno uv), no se reinstala. En Colab reemplaza la versión preinstalada.
- **Librerías:** `pymongo` y `requests` se instalan solo si faltan.
- **Java:** se reutiliza el primer JDK 17 o superior que se encuentre (`JAVA_HOME`, `java` del PATH o rutas habituales). Si no hay ninguno, se instala OpenJDK 17 con `apt-get` en Colab o Linux con root, o Temurin 17 con `install-jdk`, que no requiere administrador.
- **Variables:** `JAVA_HOME` (y su `bin` en el PATH), `PYSPARK_PYTHON`, `PYSPARK_DRIVER_PYTHON` y `SPARK_LOCAL_IP`.

In [1]:
import os, sys, platform, shutil, subprocess, re, glob, importlib.util, importlib.metadata
from pathlib import Path

VERSION_PYSPARK = "4.3.0"
JAVA_MINIMA = 17
SISTEMA = platform.system()
EJECUTABLE_JAVA = "java.exe" if SISTEMA == "Windows" else "java"
EN_COLAB = "google.colab" in sys.modules


def instalar_paquete(paquete):
    """Instala un paquete en el intérprete actual, con pip o, en entornos uv sin pip, con uv."""
    if importlib.util.find_spec("pip"):
        comando = [sys.executable, "-m", "pip", "install", "-q", paquete]
    elif shutil.which("uv"):
        comando = ["uv", "pip", "install", "-q", "--python", sys.executable, paquete]
    else:
        raise RuntimeError(f"No hay pip ni uv para instalar {paquete}")
    subprocess.run(comando, check=True)


def version_instalada(distribucion):
    try:
        return importlib.metadata.version(distribucion)
    except importlib.metadata.PackageNotFoundError:
        return None


# --- Spark y librerías de conexión ---------------------------------------
if version_instalada("pyspark") != VERSION_PYSPARK:
    print(f"Se instala pyspark=={VERSION_PYSPARK} (versión actual: {version_instalada('pyspark')})")
    instalar_paquete(f"pyspark=={VERSION_PYSPARK}")
for paquete in ["pymongo", "requests"]:
    if version_instalada(paquete) is None:
        print("Se instala", paquete)
        instalar_paquete(paquete)
print("Paquetes:", {p: version_instalada(p) for p in ["pyspark", "pymongo", "requests"]})


# --- Java ----------------------------------------------------------------
def version_java(java_home):
    """Versión mayor del JDK en java_home, o None si no hay un java ejecutable."""
    java = Path(java_home) / "bin" / EJECUTABLE_JAVA
    if not java.is_file():
        return None
    try:
        salida = subprocess.run([str(java), "-version"], capture_output=True, text=True, timeout=60).stderr
    except (OSError, subprocess.SubprocessError):
        return None
    coincidencia = re.search(r'version "(\d+)(?:\.(\d+))?', salida)
    if not coincidencia:
        return None
    mayor = int(coincidencia.group(1))
    return int(coincidencia.group(2) or 0) if mayor == 1 else mayor   # "1.8" corresponde a Java 8


def home_de_jdk(ruta):
    """En macOS el JDK descomprimido guarda su home en Contents/Home."""
    ruta = Path(ruta)
    return ruta / "Contents" / "Home" if (ruta / "Contents" / "Home").is_dir() else ruta


def candidatos_java():
    if os.environ.get("JAVA_HOME"):
        yield "JAVA_HOME", os.environ["JAVA_HOME"]
    if SISTEMA == "Darwin":
        consulta = subprocess.run(["/usr/libexec/java_home", "-v", f"{JAVA_MINIMA}+"], capture_output=True, text=True)
        if consulta.returncode == 0:
            yield "java_home de macOS", consulta.stdout.strip()
    java_en_path = shutil.which("java")
    if java_en_path:
        home = Path(os.path.realpath(java_en_path)).parent.parent
        if not (SISTEMA == "Darwin" and str(home) == "/usr"):   # lanzador del sistema, no un JDK
            yield "PATH", home
    patrones = {
        "Windows": [r"C:\Program Files\Eclipse Adoptium\*", r"C:\Program Files\Java\*", r"C:\Program Files\Microsoft\jdk-*"],
        "Darwin": ["/Library/Java/JavaVirtualMachines/*"],
        "Linux": ["/usr/lib/jvm/*"],
    }.get(SISTEMA, []) + [str(Path.home() / ".jdk" / "*")]   # instalaciones de install-jdk
    for patron in patrones:
        for ruta in sorted(glob.glob(patron), reverse=True):
            yield "ruta habitual", ruta


def seleccionar_java():
    revisados = set()
    for origen, ruta in candidatos_java():
        java_home = home_de_jdk(os.path.realpath(ruta))
        if java_home in revisados or (SISTEMA == "Darwin" and str(java_home) == "/usr"):
            continue
        revisados.add(java_home)
        version = version_java(java_home)
        if version is not None and version >= JAVA_MINIMA:
            return origen, str(java_home), version
        print(f"  Descartado ({origen}): {java_home} -> versión {version}")
    return None


def instalar_java():
    if SISTEMA == "Linux" and shutil.which("apt-get") and os.geteuid() == 0:
        subprocess.run(["apt-get", "update", "-qq"], check=True)
        subprocess.run(["apt-get", "install", "-y", "-qq", f"openjdk-{JAVA_MINIMA}-jdk-headless"], check=True)
    else:
        if importlib.util.find_spec("jdk") is None:
            instalar_paquete("install-jdk")
        import jdk
        print("  Temurin instalado en:", jdk.install(str(JAVA_MINIMA)))


seleccion = seleccionar_java()
if seleccion is None:
    print(f"No se encontró Java {JAVA_MINIMA} o superior: se instala un JDK {JAVA_MINIMA}.")
    try:
        instalar_java()
    except Exception as error:
        raise RuntimeError(f"No fue posible instalar Java automáticamente ({error}). Instale Temurin "
                           f"{JAVA_MINIMA} desde https://adoptium.net y defina JAVA_HOME.") from error
    seleccion = seleccionar_java()
    if seleccion is None:
        raise RuntimeError("Java se instaló, pero no se encontró un JDK utilizable.")

origen_java, java_home, version_jdk = seleccion
print(f"Java {version_jdk} encontrado mediante {origen_java}.")

# --- Variables de entorno ------------------------------------------------
os.environ["JAVA_HOME"] = java_home
os.environ["PATH"] = str(Path(java_home) / "bin") + os.pathsep + os.environ["PATH"]
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
os.environ.setdefault("SPARK_LOCAL_IP", "127.0.0.1")
for variable in ["JAVA_HOME", "PYSPARK_PYTHON", "PYSPARK_DRIVER_PYTHON", "SPARK_LOCAL_IP"]:
    print(f"{variable:22s}= {os.environ[variable]}")

Paquetes: {'pyspark': '4.3.0', 'pymongo': '4.19.0', 'requests': '2.34.2'}
Java 17 encontrado mediante JAVA_HOME.
JAVA_HOME             = C:\Program Files\Eclipse Adoptium\jdk-17.0.21.9-hotspot
PYSPARK_PYTHON        = C:\DEV\magister\3. Gestion de Datos y Tecnologias Big Data\gestiondatosytecnologiasbigdata\.venv\Scripts\python.exe
PYSPARK_DRIVER_PYTHON = C:\DEV\magister\3. Gestion de Datos y Tecnologias Big Data\gestiondatosytecnologiasbigdata\.venv\Scripts\python.exe
SPARK_LOCAL_IP        = 127.0.0.1


### 1.2 SparkSession y SparkContext

La sesión se crea en modo local clásico, con la zona horaria `America/Santiago` para que las fechas se muestren en hora de Chile. El `SparkContext` se obtiene de la misma sesión. Una prueba mínima confirma que el motor responde antes de conectar las fuentes.

In [2]:
import pyspark
from pyspark.sql import SparkSession, functions as F, types as T

spark = (SparkSession.builder
         .master("local[2]")
         .appName("mcdi502_s3_grupo1")
         .config("spark.sql.session.timeZone", "America/Santiago")
         .config("spark.sql.shuffle.partitions", "2")
         .config("spark.ui.showConsoleProgress", "false")
         .getOrCreate())
sc = spark.sparkContext
sc.setLogLevel("ERROR")

print("Python      :", platform.python_version())
print("PySpark     :", pyspark.__version__)
print("Spark       :", spark.version)
print("Java        :", sc._jvm.java.lang.System.getProperty("java.version"))
print("Master      :", sc.master, "· aplicación:", sc.appName)
print("Zona horaria:", spark.conf.get("spark.sql.session.timeZone"))
assert spark.range(10).count() == 10
print("Arranque de Spark verificado")

Python      : 3.14.4
PySpark     : 4.3.0
Spark       : 4.3.0
Java        : 17.0.21
Master      : local[2] · aplicación: mcdi502_s3_grupo1
Zona horaria: America/Santiago


Arranque de Spark verificado


### 1.3 Datos de acceso y parámetros comunes

Esta celda reúne los datos de acceso a las fuentes: la URI de MongoDB entregada por el docente, el token gratuito de AQICN y la URL del endpoint de MySQL. Se tratan como parámetros de configuración del taller, de modo que el notebook se ejecuta completo sin pedir datos. Si están definidas las variables de entorno `MONGODB_URI` o `AQICN_TOKEN`, se usan esas. Si un valor de la celda está vacío y no hay variable, se busca en un archivo de la carpeta `.local/` (`mongodb.json` con `{"uri": ...}` y `aqicn.json` con `{"token": ...}`).

Ambas API usan la misma ciudad, **Santiago de Chile**, y las fechas se interpretan en `America/Santiago`.

In [3]:
import json, time, math
from datetime import datetime, timezone
import requests

# --- Datos de acceso a las fuentes ------------------------------------------
MONGODB_URI = ""
AQICN_TOKEN = ""
MYSQL_URL = "http://mgtt.comunidadingenieria.cl/obtener_datos.php"


def dato_de_acceso(variable, valor, archivo, clave):
    """Variable de entorno, valor de esta celda o archivo en .local/, en ese orden."""
    valor = os.environ.get(variable) or valor
    for base in [Path.cwd(), *Path.cwd().parents]:
        if valor:
            break
        for ruta in (base / ".local" / archivo, base / "sumativa3" / ".local" / archivo):
            if ruta.is_file():
                valor = json.loads(ruta.read_text(encoding="utf-8")).get(clave)
                break
    if not valor:
        raise ValueError(f"Falta {variable}: completar esta celda o definir la variable de entorno.")
    return valor.strip()


mongo_uri = dato_de_acceso("MONGODB_URI", MONGODB_URI, "mongodb.json", "uri")
aqicn_token = dato_de_acceso("AQICN_TOKEN", AQICN_TOKEN, "aqicn.json", "token")

# --- Parámetros comunes -----------------------------------------------------
CIUDAD = "Santiago"
LATITUD, LONGITUD = -33.4489, -70.6693        # Plaza de Armas de Santiago
ZONA_HORARIA = "America/Santiago"
REGISTROS = 100
fecha_extraccion_utc = datetime.now(timezone.utc)
print("Fuentes configuradas: OpenMeteo, AQICN, MongoDB y MySQL")
print(f"Ciudad: {CIUDAD} ({LATITUD}, {LONGITUD}) · extracción UTC: {fecha_extraccion_utc:%Y-%m-%d %H:%M:%S}")

Fuentes configuradas: OpenMeteo, AQICN, MongoDB y MySQL
Ciudad: Santiago (-33.4489, -70.6693) · extracción UTC: 2026-11-14 03:14:34


## 2. Ingesta de datos y creación de DataFrames (ID3.1)

Cada fuente produce un DataFrame de **100 registros** con las variables solicitadas, tipos explícitos, la fecha original de cada registro y una clave que identifica cada fila. La misma función de control verifica todas las fuentes: 100 filas, clave única y sin nulos, nulos por columna y cobertura temporal. Los nulos se informan, pero no se imputan.

In [4]:
def validar_fuente(nombre, df, claves, columna_fecha, filas=REGISTROS):
    """Controles de aceptación comunes a las cuatro fuentes."""
    cantidad = df.count()
    assert cantidad == filas, f"{nombre}: se requieren {filas} filas y hay {cantidad}"
    assert df.select(*claves).distinct().count() == cantidad, f"{nombre}: la clave {claves} se repite"
    assert df.filter(" OR ".join(f"{c} IS NULL" for c in claves)).count() == 0, f"{nombre}: clave nula"
    print(f"{nombre}: {cantidad} filas · clave única {claves}")
    df.printSchema()
    print("Nulos por columna:")
    df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns]).show(truncate=False)
    df.select(F.min(columna_fecha).alias("fecha_min"), F.max(columna_fecha).alias("fecha_max"),
              F.countDistinct(F.to_date(columna_fecha)).alias("dias_distintos")).show(truncate=False)

### 2.1 OpenMeteo → `df_openmeteo`

- **Consulta:** se usa el endpoint `forecast` de OpenMeteo y se pide la **serie horaria de los últimos 7 días completos** (`hourly`, `past_days=7`, `forecast_days=0`) para Santiago, en `America/Santiago`. Así cada fila corresponde a una hora distinta y real, y no a una copia de la lectura actual.
- **Variables:** temperatura a 2 m (°C), humedad relativa a 2 m (%), precipitación (mm), velocidad del viento a 10 m (km/h), dirección del viento a 10 m (°) e índice UV. Las unidades se validan con `hourly_units`.
- **Selección de 100 registros:** la serie tiene 168 horas. Se eligen 100 horas repartidas de forma uniforme y determinista en los siete días (`round(i·167/99)`, i = 0…99), que incluyen la primera y la última hora. La serie completa se conserva en `df_openmeteo_serie`, como respaldo para la consulta C3 de la última semana.

In [5]:
VARIABLES_OPENMETEO = {   # nombre en la API -> columna del DataFrame
    "temperature_2m": "temperatura_2m",
    "relative_humidity_2m": "humedad_relativa_2m",
    "precipitation": "precipitacion",
    "wind_speed_10m": "velocidad_viento_10m",
    "wind_direction_10m": "direccion_viento_10m",
    "uv_index": "indice_uv",
}
UNIDADES_ESPERADAS = {"temperature_2m": "°C", "relative_humidity_2m": "%", "precipitation": "mm",
                      "wind_speed_10m": "km/h", "wind_direction_10m": "°"}

try:
    respuesta = requests.get("https://api.open-meteo.com/v1/forecast", timeout=30, params={
        "latitude": LATITUD, "longitude": LONGITUD, "hourly": ",".join(VARIABLES_OPENMETEO),
        "past_days": 7, "forecast_days": 0, "timezone": ZONA_HORARIA})
    respuesta.raise_for_status()
    datos_openmeteo = respuesta.json()
except Exception as error:
    raise RuntimeError("No se pudo consultar OpenMeteo: " + type(error).__name__) from None

horario, unidades = datos_openmeteo["hourly"], datos_openmeteo["hourly_units"]
faltan = [v for v in VARIABLES_OPENMETEO if v not in horario]
assert not faltan, f"OpenMeteo no entregó: {faltan}"
for variable, unidad in UNIDADES_ESPERADAS.items():
    assert unidades[variable] == unidad, f"Unidad inesperada para {variable}: {unidades[variable]}"
horas = [datetime.fromisoformat(h) for h in horario["time"]]
assert len(horas) == 168 and len(set(horas)) == 168, f"Se esperaban 168 horas distintas; hay {len(horas)}"

filas_openmeteo = [
    (i, hora, CIUDAD, *[None if horario[v][i] is None else float(horario[v][i]) for v in VARIABLES_OPENMETEO])
    for i, hora in enumerate(horas)
]
schema_openmeteo = T.StructType([
    T.StructField("indice_hora", T.IntegerType(), False),
    T.StructField("fecha_hora", T.TimestampNTZType(), False),   # hora local de Santiago, como la entrega la API
    T.StructField("ciudad", T.StringType(), False),
    *[T.StructField(c, T.DoubleType(), True) for c in VARIABLES_OPENMETEO.values()],
])
df_openmeteo_serie = spark.createDataFrame(filas_openmeteo, schema_openmeteo)

ultimo = len(horas) - 1
indices_muestra = sorted({round(i * ultimo / (REGISTROS - 1)) for i in range(REGISTROS)})
assert len(indices_muestra) == REGISTROS
df_openmeteo = (df_openmeteo_serie.filter(F.col("indice_hora").isin(indices_muestra))
                .orderBy("fecha_hora"))

print(f"Ubicación devuelta: {datos_openmeteo['latitude']}, {datos_openmeteo['longitude']} · zona {datos_openmeteo['timezone']}")
print(f"Serie horaria: {len(horas)} horas, del {horas[0]} al {horas[-1]} (corte de la serie)")
print("Unidades:", {VARIABLES_OPENMETEO[k]: unidades[k] for k in VARIABLES_OPENMETEO})

Ubicación devuelta: -33.427067, -70.64276 · zona America/Santiago
Serie horaria: 168 horas, del 2026-11-07 00:00:00 al 2026-11-13 23:00:00 (corte de la serie)
Unidades: {'temperatura_2m': '°C', 'humedad_relativa_2m': '%', 'precipitacion': 'mm', 'velocidad_viento_10m': 'km/h', 'direccion_viento_10m': '°', 'indice_uv': ''}


In [6]:
validar_fuente("df_openmeteo", df_openmeteo, ["fecha_hora"], "fecha_hora")
df_openmeteo.show(10, truncate=False)

df_openmeteo: 100 filas · clave única ['fecha_hora']
root
 |-- indice_hora: integer (nullable = false)
 |-- fecha_hora: timestamp_ntz (nullable = false)
 |-- ciudad: string (nullable = false)
 |-- temperatura_2m: double (nullable = true)
 |-- humedad_relativa_2m: double (nullable = true)
 |-- precipitacion: double (nullable = true)
 |-- velocidad_viento_10m: double (nullable = true)
 |-- direccion_viento_10m: double (nullable = true)
 |-- indice_uv: double (nullable = true)

Nulos por columna:


+-----------+----------+------+--------------+-------------------+-------------+--------------------+--------------------+---------+
|indice_hora|fecha_hora|ciudad|temperatura_2m|humedad_relativa_2m|precipitacion|velocidad_viento_10m|direccion_viento_10m|indice_uv|
+-----------+----------+------+--------------+-------------------+-------------+--------------------+--------------------+---------+
|0          |0         |0     |0             |0                  |0            |0                   |0                   |0        |
+-----------+----------+------+--------------+-------------------+-------------+--------------------+--------------------+---------+



+-------------------+-------------------+--------------+
|fecha_min          |fecha_max          |dias_distintos|
+-------------------+-------------------+--------------+
|2026-11-07 00:00:00|2026-11-13 23:00:00|7             |
+-------------------+-------------------+--------------+



+-----------+-------------------+--------+--------------+-------------------+-------------+--------------------+--------------------+---------+
|indice_hora|fecha_hora         |ciudad  |temperatura_2m|humedad_relativa_2m|precipitacion|velocidad_viento_10m|direccion_viento_10m|indice_uv|
+-----------+-------------------+--------+--------------+-------------------+-------------+--------------------+--------------------+---------+
|0          |2026-11-07 00:00:00|Santiago|13.0          |49.0               |0.0          |5.2                 |25.0                |0.0      |
|2          |2026-11-07 02:00:00|Santiago|11.5          |60.0               |0.0          |2.3                 |51.0                |0.0      |
|3          |2026-11-07 03:00:00|Santiago|10.9          |65.0               |0.0          |1.9                 |139.0               |0.0      |
|5          |2026-11-07 05:00:00|Santiago|9.8           |75.0               |0.0          |0.9                 |53.0                |0.0

### 2.2 AQICN → `df_aqicn`

- **Token:** el token público `demo` devuelve siempre los datos de **Shanghái**, aunque se pida otra ciudad. Por eso se usa un token personal (gratuito) y se rechaza cualquier respuesta cuyas coordenadas no estén en Chile.
- **Por qué no repetir la consulta:** el feed de una ciudad entrega solo la última lectura horaria de una estación. Repetirlo 100 veces produce 100 copias del mismo valor.
- **Estrategia:** se buscan las estaciones de la región con `map/bounds`, comenzando por la Región Metropolitana. Si no alcanzan, la búsqueda se amplía a la zona central y luego a Chile. Las estaciones se ordenan por distancia a Santiago (desempate por `uid`) y se consulta el feed de cada una (`feed/@uid`). La primera ronda entrega una observación real por estación chilena con contaminantes.
- **Variables:** PM2.5, PM10, O3, NO2 y SO2 desde `iaqi`. AQICN entrega cada contaminante como **sub-índice AQI (escala US EPA)**, no como concentración en µg/m³. Las consultas se interpretan en esa escala. Algunas estaciones no miden todos los contaminantes: esos valores quedan nulos y se informan.
- **Completar las 100 filas:** AQICN publica solo unas decenas de estaciones para todo Chile, menos de las 100 requeridas (la cantidad exacta se imprime abajo). Por eso, después de la primera ronda se repite la consulta a las mismas estaciones, en el mismo orden, hasta completar 100 capturas. Cada captura repetida queda identificada en la columna `ronda` y conserva su `fecha_captura`; la clave de la fila es (`estacion_uid`, `ronda`). No se inventan ni se imputan valores.
- **Vigencia:** cada fila guarda `fecha_observacion`, la hora de la lectura según la estación. Si la red de monitoreo no ha publicado datos recientes, las rondas repiten la misma lectura. La celda de control informa la antigüedad de las observaciones para que esto quede a la vista.

In [7]:
CONTAMINANTES = {"pm25": "pm25", "pm10": "pm10", "o3": "o3", "no2": "no2", "so2": "so2"}
CAJAS_BUSQUEDA = [   # (nombre, (lat_sur, lon_oeste, lat_norte, lon_este)), de menor a mayor
    ("Región Metropolitana", (-34.30, -71.75, -32.90, -69.75)),
    ("Zona central", (-36.60, -72.90, -32.00, -69.75)),
    ("Chile", (-56.00, -76.00, -17.50, -66.00)),
]


SESION_AQICN = requests.Session()   # reutiliza la conexión HTTPS: ~3 veces más rápido que abrir una por consulta


def consultar_aqicn(ruta, **parametros):
    """GET a AQICN sin exponer el token en errores ni salidas."""
    try:
        respuesta = SESION_AQICN.get(f"https://api.waqi.info{ruta}", timeout=30,
                                 params={**parametros, "token": aqicn_token})
        respuesta.raise_for_status()
        datos = respuesta.json()
    except Exception as error:
        raise RuntimeError(f"No se pudo consultar AQICN {ruta.split('@')[0]}: {type(error).__name__}") from None
    if datos.get("status") != "ok":
        detalle = datos.get("data") if isinstance(datos.get("data"), str) else "sin detalle"
        raise RuntimeError(f"AQICN respondió '{datos.get('status')}': {detalle}")
    return datos["data"]


def distancia_km(lat, lon):
    """Distancia de Haversine a Santiago."""
    p1, p2 = math.radians(LATITUD), math.radians(lat)
    dp, dl = p2 - p1, math.radians(lon - LONGITUD)
    return 6371 * 2 * math.asin(math.sqrt(math.sin(dp / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(dl / 2) ** 2))


def es_de_chile(feed):
    ciudad = feed.get("city", {})
    lat = (ciudad.get("geo") or [0, 0])[0]
    texto = f"{ciudad.get('url', '')} {ciudad.get('name', '')}".lower()
    return -56 <= lat <= -17 and "chile" in texto


# 1) Estaciones candidatas, ampliando la búsqueda solo si hace falta
candidatas, region_usada = {}, None
for region, caja in CAJAS_BUSQUEDA:
    for estacion in consultar_aqicn("/map/bounds/", latlng=",".join(map(str, caja))):
        candidatas[estacion["uid"]] = estacion      # aqi "-" = sin índice vigente; el feed puede traer contaminantes
    region_usada = region
    vigentes = sum(str(e.get("aqi", "")).strip().isdigit() for e in candidatas.values())
    print(f"{region}: {len(candidatas)} estaciones ({vigentes} con AQI vigente)")
    if len(candidatas) >= REGISTROS * 1.3:                          # margen para descartes
        break
orden = sorted(candidatas.values(), key=lambda e: (distancia_km(e["lat"], e["lon"]), e["uid"]))

# 2) Feed de cada estación, de la más cercana a la más lejana
filas_aqicn, descartadas = [], {"fuera de Chile": 0, "sin contaminantes": 0, "error": 0}
for estacion in orden:
    if len(filas_aqicn) == REGISTROS:
        break
    try:
        feed = consultar_aqicn(f"/feed/@{estacion['uid']}/")
    except RuntimeError:
        descartadas["error"] += 1
        continue
    if not es_de_chile(feed):
        descartadas["fuera de Chile"] += 1
        continue
    iaqi = feed.get("iaqi", {})
    valores = [None if iaqi.get(c, {}).get("v") is None else float(iaqi[c]["v"]) for c in CONTAMINANTES]
    if all(v is None for v in valores):
        descartadas["sin contaminantes"] += 1
        continue
    lat, lon = feed["city"]["geo"][:2]
    filas_aqicn.append((int(estacion["uid"]), feed["city"]["name"], float(lat), float(lon),
                        round(distancia_km(lat, lon), 1), datetime.fromisoformat(feed["time"]["iso"]),
                        datetime.now(timezone.utc), 1, *valores))
    time.sleep(0.2)                                                   # cortesía con la API

# 3) Respaldo documentado: rondas adicionales solo si no hubo 100 estaciones
estaciones_distintas = len(filas_aqicn)
ronda = 1
while 0 < len(filas_aqicn) < REGISTROS:
    ronda += 1
    for fila in filas_aqicn[:estaciones_distintas]:
        if len(filas_aqicn) == REGISTROS:
            break
        feed = consultar_aqicn(f"/feed/@{fila[0]}/")
        iaqi = feed.get("iaqi", {})
        valores = [None if iaqi.get(c, {}).get("v") is None else float(iaqi[c]["v"]) for c in CONTAMINANTES]
        filas_aqicn.append((*fila[:5], datetime.fromisoformat(feed["time"]["iso"]),
                            datetime.now(timezone.utc), ronda, *valores))
        time.sleep(0.5)

schema_aqicn = T.StructType([
    T.StructField("estacion_uid", T.IntegerType(), False),
    T.StructField("estacion", T.StringType(), False),
    T.StructField("latitud", T.DoubleType(), False),
    T.StructField("longitud", T.DoubleType(), False),
    T.StructField("distancia_km", T.DoubleType(), False),
    T.StructField("fecha_observacion", T.TimestampType(), False),   # instante con zona, mostrado en America/Santiago
    T.StructField("fecha_captura", T.TimestampType(), False),
    T.StructField("ronda", T.IntegerType(), False),
    *[T.StructField(c, T.DoubleType(), True) for c in CONTAMINANTES.values()],
])
df_aqicn = spark.createDataFrame(filas_aqicn, schema_aqicn).orderBy("distancia_km", "estacion_uid", "ronda")

print(f"Búsqueda hasta: {region_usada} · estaciones distintas: {estaciones_distintas} · rondas: {ronda}")
print("Estaciones descartadas:", descartadas)

Región Metropolitana: 15 estaciones (0 con AQI vigente)


Zona central: 29 estaciones (0 con AQI vigente)


Chile: 50 estaciones (0 con AQI vigente)


Búsqueda hasta: Chile · estaciones distintas: 50 · rondas: 2
Estaciones descartadas: {'fuera de Chile': 0, 'sin contaminantes': 0, 'error': 0}


In [8]:
validar_fuente("df_aqicn", df_aqicn, ["estacion_uid", "ronda"], "fecha_observacion")
print("Estaciones distintas:", df_aqicn.select("estacion_uid").distinct().count(),
      "· observaciones distintas (estación, hora de observación):",
      df_aqicn.select("estacion_uid", "fecha_observacion").distinct().count())
print("Capturas por ronda:")
df_aqicn.groupBy("ronda").count().orderBy("ronda").show()
print("Antigüedad de las observaciones al momento de la captura (horas):")
df_aqicn.select(((F.unix_timestamp("fecha_captura") - F.unix_timestamp("fecha_observacion")) / 3600)
                .alias("horas")).agg(F.round(F.min("horas"), 1).alias("minima"),
                                     F.round(F.max("horas"), 1).alias("maxima")).show()
print("Valores válidos por contaminante (escala AQI US EPA):")
df_aqicn.select([F.count(c).alias(c) for c in CONTAMINANTES.values()]).show()
df_aqicn.select("estacion_uid", "estacion", "distancia_km", "fecha_observacion",
                *CONTAMINANTES.values()).show(10, truncate=40)

df_aqicn: 100 filas · clave única ['estacion_uid', 'ronda']
root
 |-- estacion_uid: integer (nullable = false)
 |-- estacion: string (nullable = false)
 |-- latitud: double (nullable = false)
 |-- longitud: double (nullable = false)
 |-- distancia_km: double (nullable = false)
 |-- fecha_observacion: timestamp (nullable = false)
 |-- fecha_captura: timestamp (nullable = false)
 |-- ronda: integer (nullable = false)
 |-- pm25: double (nullable = true)
 |-- pm10: double (nullable = true)
 |-- o3: double (nullable = true)
 |-- no2: double (nullable = true)
 |-- so2: double (nullable = true)

Nulos por columna:


+------------+--------+-------+--------+------------+-----------------+-------------+-----+----+----+---+---+---+
|estacion_uid|estacion|latitud|longitud|distancia_km|fecha_observacion|fecha_captura|ronda|pm25|pm10|o3 |no2|so2|
+------------+--------+-------+--------+------------+-----------------+-------------+-----+----+----+---+---+---+
|0           |0       |0      |0       |0           |0                |0            |0    |2   |26  |68 |66 |74 |
+------------+--------+-------+--------+------------+-----------------+-------------+-----+----+----+---+---+---+



+-------------------+-------------------+--------------+
|fecha_min          |fecha_max          |dias_distintos|
+-------------------+-------------------+--------------+
|2026-11-09 13:00:00|2026-11-09 21:00:00|1             |
+-------------------+-------------------+--------------+



Estaciones distintas: 50 · observaciones distintas (estación, hora de observación): 50
Capturas por ronda:


+-----+-----+
|ronda|count|
+-----+-----+
|    1|   50|
|    2|   50|
+-----+-----+

Antigüedad de las observaciones al momento de la captura (horas):


+------+------+
|minima|maxima|
+------+------+
|  99.2| 107.3|
+------+------+

Valores válidos por contaminante (escala AQI US EPA):


+----+----+---+---+---+
|pm25|pm10| o3|no2|so2|
+----+----+---+---+---+
|  98|  74| 32| 34| 26|
+----+----+---+---+---+



+------------+-----------------------+------------+-------------------+----+----+----+----+----+
|estacion_uid|               estacion|distancia_km|  fecha_observacion|pm25|pm10|  o3| no2| so2|
+------------+-----------------------+------------+-------------------+----+----+----+----+----+
|         431|Parque O'Higgins, Chile|         1.9|2026-11-09 20:00:00|25.0|29.0|18.4|12.7|NULL|
|         431|Parque O'Higgins, Chile|         1.9|2026-11-09 20:00:00|25.0|29.0|18.4|12.7|NULL|
|         435|     Cerro Navia, Chile|         6.1|2026-11-09 21:00:00|50.0|26.0| 8.8|11.7|NULL|
|         435|     Cerro Navia, Chile|         6.1|2026-11-09 21:00:00|50.0|26.0| 8.8|11.7|NULL|
|       14731|    Cerrillos II, Chile|         6.1|2026-11-09 20:00:00|25.0|38.0|NULL|NULL|NULL|
|       14731|    Cerrillos II, Chile|         6.1|2026-11-09 20:00:00|25.0|38.0|NULL|NULL|NULL|
|         432|        Pudahuel, Chile|         7.6|2026-11-09 20:00:00|42.0|37.0|10.4|14.9|NULL|
|         432|        Pudahuel

### 2.3 MongoDB (ESP32) → `df_mongo_bpm`

- **Origen:** colección `telemetria.lecturas` proporcionada por el docente, leída con PyMongo en modo lectura.
- **Selección:** las 100 lecturas más recientes, ordenadas por `fecha` descendente y `_id` descendente. El orden se aplica en el servidor antes de limitar.
- **Mapeo de campos:** `altitudFt` → `alturaft`, `presionAtm` → `presion_atmosferica`, `ppmCO` → `monoxido_carbono`. También se conserva `presionPa` como `presion_pa`. Los nombres sugieren atm, ppm y Pa como unidades.
- **Fechas:** las fechas BSON son instantes UTC, y Spark las muestra en `America/Santiago`.
- **Tipado:** los valores se reciben como texto y se convierten con `try_cast`. El conteo de valores no convertibles se informa.

In [9]:
from pymongo import MongoClient


def texto_o_nulo(valor):
    return None if valor is None else str(valor)

globals().pop('df_mongo_bpm', None)
try:
    with MongoClient(mongo_uri, serverSelectionTimeoutMS=15000,
                     connectTimeoutMS=15000, socketTimeoutMS=30000,
                     tz_aware=True) as cliente:
        documentos = list(cliente['telemetria']['lecturas'].find(
            {}, {'altitudFt': 1, 'presionAtm': 1, 'presionPa': 1,
                 'ppmCO': 1, 'fecha': 1}
        ).sort([('fecha', -1), ('_id', -1)]).limit(100))
except Exception as error:
    # No exponer la URI ni el mensaje del servidor en las salidas.
    raise RuntimeError('No se pudo leer MongoDB: ' + type(error).__name__ +
                       '. Revisar acceso, red y URI.') from None

if len(documentos) != 100:
    raise ValueError(f'MongoDB devolvió {len(documentos)} registros; se requieren 100.')
campos_mongo = {'_id', 'fecha', 'altitudFt', 'presionAtm', 'ppmCO'}
if not campos_mongo.issubset(set().union(*(d.keys() for d in documentos))):
    raise ValueError('La colección no contiene todos los campos solicitados.')
if any(not isinstance(d.get('fecha'), datetime) for d in documentos):
    raise ValueError('Revisar fechas BSON faltantes o no válidas antes de seleccionar filas.')

filas_mongo = [(
    str(d['_id']), d['fecha'], texto_o_nulo(d.get('altitudFt')),
    texto_o_nulo(d.get('presionAtm')), texto_o_nulo(d.get('presionPa')),
    texto_o_nulo(d.get('ppmCO'))
) for d in documentos]
schema_mongo = T.StructType([
    T.StructField('id', T.StringType(), False),
    T.StructField('fecha', T.TimestampType(), False),
    *[T.StructField(c, T.StringType(), True) for c in
      ['alturaft', 'presion_atmosferica', 'presion_pa', 'monoxido_carbono']]
])
mongo_original = spark.createDataFrame(filas_mongo, schema_mongo)
df_mongo_bpm = mongo_original
for campo in ['alturaft', 'presion_atmosferica', 'presion_pa', 'monoxido_carbono']:
    fallos = df_mongo_bpm.filter(
        F.col(campo).isNotNull() & F.expr(f'try_cast({campo} AS DOUBLE)').isNull()
    ).count()
    print(f'MongoDB {campo}: valores no convertibles = {fallos}')
    df_mongo_bpm = df_mongo_bpm.withColumn(campo, F.expr(f'try_cast({campo} AS DOUBLE)'))
df_mongo_bpm = df_mongo_bpm.orderBy(F.col('fecha').desc(), F.col('id').desc())
# altura_metros se calcula en la sección 3.1

MongoDB alturaft: valores no convertibles = 0


MongoDB presion_atmosferica: valores no convertibles = 0


MongoDB presion_pa: valores no convertibles = 0


MongoDB monoxido_carbono: valores no convertibles = 0


### 2.4 MySQL (ESP32) → `df_mysql_dht`

- **Origen:** endpoint PHP del curso (`obtener_datos.php`), que expone la tabla MySQL como JSON.
- **Variables:** temperatura (°C), que se renombra `temperatura_interior` según el enunciado, y humedad (%).
- **Fechas:** se conservan `fecha_original` (el texto recibido) y `fecha` (`TIMESTAMP_NTZ`), porque la fuente no declara zona horaria.
- **Limpieza:** se eliminan los duplicados exactos y se rechazan los IDs con registros contradictorios. Luego se toman los 100 más recientes por `fecha` e `id`.

In [10]:
try:
    respuesta = requests.get(MYSQL_URL, timeout=30)
    respuesta.raise_for_status()
    datos_mysql = respuesta.json()
except Exception as error:
    raise RuntimeError('No se pudo leer MySQL/REST: ' + type(error).__name__) from None
if not isinstance(datos_mysql, list) or not datos_mysql:
    raise ValueError('Se esperaba una lista JSON no vacía de registros.')
campos_mysql = {'id', 'temperatura', 'humedad', 'fecha'}
if any(not isinstance(d, dict) or not campos_mysql.issubset(d) for d in datos_mysql):
    raise ValueError('El JSON no coincide con el esquema esperado.')

filas_mysql = [tuple(texto_o_nulo(d[c]) for c in ['id','temperatura','humedad','fecha'])
               for d in datos_mysql]
mysql_original = spark.createDataFrame(
    filas_mysql, 'id_original STRING, temperatura_interior STRING, humedad STRING, fecha_original STRING')
df_mysql_tipado = mysql_original
for campo, tipo in [('id_original', 'BIGINT'), ('temperatura_interior', 'DOUBLE'),
                    ('humedad', 'DOUBLE'), ('fecha_original', 'TIMESTAMP_NTZ')]:
    conversion = F.expr(f'try_cast({campo} AS {tipo})')
    fallos = df_mysql_tipado.filter(F.col(campo).isNotNull() & conversion.isNull()).count()
    print(f'MySQL {campo}: valores no convertibles = {fallos}')
    destino = {'id_original': 'id', 'fecha_original': 'fecha'}.get(campo, campo)
    df_mysql_tipado = df_mysql_tipado.withColumn(destino, conversion)
if df_mysql_tipado.filter(F.col('id').isNull() | F.col('fecha').isNull()).count():
    raise ValueError('Hay IDs o fechas no válidos; corregir antes de ordenar y seleccionar.')
sin_duplicados = df_mysql_tipado.dropDuplicates()
if sin_duplicados.groupBy('id').count().filter(F.col('count') > 1).count():
    raise ValueError('Hay IDs con registros contradictorios; revisar la fuente.')
print('Filas recibidas:', len(datos_mysql))
print('Duplicados exactos eliminados:', df_mysql_tipado.count() - sin_duplicados.count())
df_mysql_dht = (sin_duplicados.orderBy(F.col('fecha').desc(), F.col('id').desc())
                .limit(100).drop('id_original'))

MySQL id_original: valores no convertibles = 0


MySQL temperatura_interior: valores no convertibles = 0


MySQL humedad: valores no convertibles = 0


MySQL fecha_original: valores no convertibles = 0


Filas recibidas: 1000


Duplicados exactos eliminados: 0


In [11]:
validar_fuente("df_mongo_bpm", df_mongo_bpm, ["id"], "fecha")
validar_fuente("df_mysql_dht", df_mysql_dht, ["id"], "fecha")
df_mysql_dht.show(10, truncate=False)

df_mongo_bpm: 100 filas · clave única ['id']
root
 |-- id: string (nullable = false)
 |-- fecha: timestamp (nullable = false)
 |-- alturaft: double (nullable = true)
 |-- presion_atmosferica: double (nullable = true)
 |-- presion_pa: double (nullable = true)
 |-- monoxido_carbono: double (nullable = true)

Nulos por columna:


+---+-----+--------+-------------------+----------+----------------+
|id |fecha|alturaft|presion_atmosferica|presion_pa|monoxido_carbono|
+---+-----+--------+-------------------+----------+----------------+
|0  |0    |0       |0                  |0         |0               |
+---+-----+--------+-------------------+----------+----------------+



+-----------------------+-----------------------+--------------+
|fecha_min              |fecha_max              |dias_distintos|
+-----------------------+-----------------------+--------------+
|2026-11-13 23:48:10.566|2026-11-14 00:16:26.706|2             |
+-----------------------+-----------------------+--------------+



df_mysql_dht: 100 filas · clave única ['id']
root
 |-- temperatura_interior: double (nullable = true)
 |-- humedad: double (nullable = true)
 |-- fecha_original: string (nullable = true)
 |-- id: long (nullable = true)
 |-- fecha: timestamp_ntz (nullable = true)

Nulos por columna:


+--------------------+-------+--------------+---+-----+
|temperatura_interior|humedad|fecha_original|id |fecha|
+--------------------+-------+--------------+---+-----+
|0                   |0      |0             |0  |0    |
+--------------------+-------+--------------+---+-----+



+-------------------+-------------------+--------------+
|fecha_min          |fecha_max          |dias_distintos|
+-------------------+-------------------+--------------+
|2026-11-13 18:48:10|2026-11-13 19:16:26|1             |
+-------------------+-------------------+--------------+



+--------------------+-------+-------------------+------+-------------------+
|temperatura_interior|humedad|fecha_original     |id    |fecha              |
+--------------------+-------+-------------------+------+-------------------+
|23.5                |63.4   |2026-11-13 19:16:26|373131|2026-11-13 19:16:26|
|23.5                |63.4   |2026-11-13 19:16:09|373130|2026-11-13 19:16:09|
|23.5                |63.4   |2026-11-13 19:15:52|373129|2026-11-13 19:15:52|
|23.6                |63.4   |2026-11-13 19:15:35|373128|2026-11-13 19:15:35|
|23.5                |63.5   |2026-11-13 19:15:18|373127|2026-11-13 19:15:18|
|23.6                |63.5   |2026-11-13 19:15:01|373126|2026-11-13 19:15:01|
|23.5                |63.5   |2026-11-13 19:14:43|373125|2026-11-13 19:14:43|
|23.5                |63.6   |2026-11-13 19:14:26|373124|2026-11-13 19:14:26|
|23.5                |63.7   |2026-11-13 19:14:09|373123|2026-11-13 19:14:09|
|23.5                |63.6   |2026-11-13 19:13:52|373122|2026-11

## 3. Preprocesamiento y transformación de datos (RA4, ID4.2)

### 3.1 `altura_metros` en `df_mongo_bpm`

`alturaft` se convierte a número y se multiplica por 0,3048 (1 pie = 0,3048 m). Los valores faltantes o no convertibles quedan como nulos y no se imputan. Antes de aplicarla a los datos reales, la función se prueba con casos aislados: 10 pies, un valor faltante y un texto no numérico.

In [12]:
def agregar_altura_metros(df):
    return (df.withColumn('alturaft', F.expr('try_cast(alturaft AS DOUBLE)'))
              .withColumn('altura_metros', F.col('alturaft') * F.lit(0.3048)))

# Casos de prueba aislados; no se utilizan para completar las 100 filas reales.
prueba_altura = spark.createDataFrame(
    [('conocido', '10'), ('faltante', None), ('invalido', 'sin dato')],
    'caso STRING, alturaft STRING'
)
resultado_prueba = agregar_altura_metros(prueba_altura)
valores = {fila.caso: fila.altura_metros for fila in resultado_prueba.collect()}
assert abs(valores['conocido'] - 3.048) < 1e-9
assert valores['faltante'] is None
assert valores['invalido'] is None
resultado_prueba.show()
print('Conversión de altura verificada')


df_mongo_bpm = agregar_altura_metros(df_mongo_bpm).orderBy(F.col("fecha").desc(), F.col("id").desc())
assert df_mongo_bpm.count() == REGISTROS
assert df_mongo_bpm.filter(
    ~F.col("altura_metros").eqNullSafe(F.col("alturaft") * F.lit(0.3048))).count() == 0
print("altura_metros aplicada a las", REGISTROS, "filas de df_mongo_bpm")
df_mongo_bpm.select("id", "fecha", "alturaft",
                    F.round("altura_metros", 3).alias("altura_metros")).show(5, truncate=False)

+--------+--------+-------------+
|    caso|alturaft|altura_metros|
+--------+--------+-------------+
|conocido|    10.0|        3.048|
|faltante|    NULL|         NULL|
|invalido|    NULL|         NULL|
+--------+--------+-------------+

Conversión de altura verificada


altura_metros aplicada a las 100 filas de df_mongo_bpm


+------------------------+-----------------------+--------+-------------+
|id                      |fecha                  |alturaft|altura_metros|
+------------------------+-----------------------+--------+-------------+
|6af7d28ac1ab6b5957d4c32a|2026-11-14 00:16:26.706|877.07  |267.331      |
|6af7d279c1ab6b5957d4c328|2026-11-14 00:16:09.592|877.63  |267.502      |
|6af7d268c1ab6b5957d4c326|2026-11-14 00:15:52.638|877.35  |267.416      |
|6af7d257c1ab6b5957d4c324|2026-11-14 00:15:35.542|877.63  |267.502      |
|6af7d246c1ab6b5957d4c322|2026-11-14 00:15:18.433|877.07  |267.331      |
+------------------------+-----------------------+--------+-------------+
only showing top 5 rows


### 3.2 `condicion_clima` en `df_openmeteo`

Las reglas se evalúan **en el orden del enunciado** con `when` encadenados, y gana la primera que se cumple:

| Regla | Resultado | Caso de prueba (frontera) |
|---|---|---|
| Temperatura > 30 °C y humedad > 70 % | CALOR Y HUMEDAD | 31 °C y 71 % |
| Temperatura > 30 °C y humedad ≤ 70 % | CALOR SECO | 31 °C y 70 % |
| Temperatura ≤ 30 °C | CLIMA MODERADO | 30 °C, incluso con humedad nula |
| Cualquier otro caso | CONDICION DESCONOCIDA | 31 °C con humedad nula; temperatura nula |

En Spark, una comparación con un valor nulo no es verdadera. Por eso, un registro con temperatura nula, o con más de 30 °C y humedad nula, no cumple ninguna de las tres primeras reglas y queda como CONDICION DESCONOCIDA en vez de clasificarse mal.

In [13]:
def agregar_condicion_clima(df, temperatura="temperatura_2m", humedad="humedad_relativa_2m"):
    t, h = F.col(temperatura), F.col(humedad)
    return df.withColumn(
        "condicion_clima",
        F.when((t > 30) & (h > 70), "CALOR Y HUMEDAD")
         .when((t > 30) & (h <= 70), "CALOR SECO")
         .when(t <= 30, "CLIMA MODERADO")
         .otherwise("CONDICION DESCONOCIDA"))


# Casos de prueba aislados; no se agregan a las 100 filas reales.
casos = [(31.0, 71.0, "CALOR Y HUMEDAD"), (31.0, 70.0, "CALOR SECO"), (30.0, None, "CLIMA MODERADO"),
         (12.5, 95.0, "CLIMA MODERADO"), (31.0, None, "CONDICION DESCONOCIDA"),
         (None, 50.0, "CONDICION DESCONOCIDA")]
prueba_clima = agregar_condicion_clima(spark.createDataFrame(
    casos, "temperatura_2m DOUBLE, humedad_relativa_2m DOUBLE, esperado STRING"))
prueba_clima.show(truncate=False)
assert prueba_clima.filter(F.col("condicion_clima") != F.col("esperado")).count() == 0
print("Las cuatro ramas de condicion_clima pasaron sus pruebas")

df_openmeteo = agregar_condicion_clima(df_openmeteo).orderBy("fecha_hora")
assert df_openmeteo.count() == REGISTROS
print("Distribución en las", REGISTROS, "horas de df_openmeteo:")
df_openmeteo.groupBy("condicion_clima").agg(
    F.count("*").alias("horas"), F.round(F.max("temperatura_2m"), 1).alias("temp_max")
).orderBy(F.desc("horas")).show(truncate=False)
df_openmeteo.select("fecha_hora", "temperatura_2m", "humedad_relativa_2m", "condicion_clima").show(10, truncate=False)

+--------------+-------------------+---------------------+---------------------+
|temperatura_2m|humedad_relativa_2m|esperado             |condicion_clima      |
+--------------+-------------------+---------------------+---------------------+
|31.0          |71.0               |CALOR Y HUMEDAD      |CALOR Y HUMEDAD      |
|31.0          |70.0               |CALOR SECO           |CALOR SECO           |
|30.0          |NULL               |CLIMA MODERADO       |CLIMA MODERADO       |
|12.5          |95.0               |CLIMA MODERADO       |CLIMA MODERADO       |
|31.0          |NULL               |CONDICION DESCONOCIDA|CONDICION DESCONOCIDA|
|NULL          |50.0               |CONDICION DESCONOCIDA|CONDICION DESCONOCIDA|
+--------------+-------------------+---------------------+---------------------+



Las cuatro ramas de condicion_clima pasaron sus pruebas


Distribución en las 100 horas de df_openmeteo:


+---------------+-----+--------+
|condicion_clima|horas|temp_max|
+---------------+-----+--------+
|CLIMA MODERADO |92   |29.9    |
|CALOR SECO     |8    |32.2    |
+---------------+-----+--------+



+-------------------+--------------+-------------------+---------------+
|fecha_hora         |temperatura_2m|humedad_relativa_2m|condicion_clima|
+-------------------+--------------+-------------------+---------------+
|2026-11-07 00:00:00|13.0          |49.0               |CLIMA MODERADO |
|2026-11-07 02:00:00|11.5          |60.0               |CLIMA MODERADO |
|2026-11-07 03:00:00|10.9          |65.0               |CLIMA MODERADO |
|2026-11-07 05:00:00|9.8           |75.0               |CLIMA MODERADO |
|2026-11-07 07:00:00|8.6           |80.0               |CLIMA MODERADO |
|2026-11-07 08:00:00|10.4          |69.0               |CLIMA MODERADO |
|2026-11-07 10:00:00|14.3          |47.0               |CLIMA MODERADO |
|2026-11-07 12:00:00|19.4          |32.0               |CLIMA MODERADO |
|2026-11-07 13:00:00|21.7          |26.0               |CLIMA MODERADO |
|2026-11-07 15:00:00|24.6          |22.0               |CLIMA MODERADO |
+-------------------+--------------+---------------

### Control previo a la integración

Esta tabla resume las cuatro fuentes ya transformadas, como punto de partida para la integración. Cada fila indica la clave, el orden documentado y la cobertura temporal de un DataFrame.

In [14]:
fuentes = {
    "df_openmeteo": (df_openmeteo, "fecha_hora", "fecha_hora asc", "hora local Santiago (API)"),
    "df_aqicn": (df_aqicn, "fecha_observacion", "distancia_km, estacion_uid, ronda", "instante con zona"),
    "df_mongo_bpm": (df_mongo_bpm, "fecha", "fecha desc, id desc", "instante UTC (BSON)"),
    "df_mysql_dht": (df_mysql_dht, "fecha", "fecha desc, id desc", "sin zona declarada"),
}
resumen = []
for nombre, (df, col_fecha, orden_doc, zona) in fuentes.items():
    extremos = df.agg(F.min(col_fecha).cast("string"), F.max(col_fecha).cast("string")).first()
    resumen.append((nombre, df.count(), len(df.columns), orden_doc, zona, extremos[0][:19], extremos[1][:19]))
    assert df.count() == REGISTROS
assert "altura_metros" in df_mongo_bpm.columns and "condicion_clima" in df_openmeteo.columns
spark.createDataFrame(resumen, "dataframe STRING, filas INT, columnas INT, orden STRING, zona_fecha STRING, "
                               "desde STRING, hasta STRING").show(truncate=False)
print("Cuatro DataFrames de 100 filas con altura_metros y condicion_clima, listos para la integración.")

+------------+-----+--------+---------------------------------+-------------------------+-------------------+-------------------+
|dataframe   |filas|columnas|orden                            |zona_fecha               |desde              |hasta              |
+------------+-----+--------+---------------------------------+-------------------------+-------------------+-------------------+
|df_openmeteo|100  |10      |fecha_hora asc                   |hora local Santiago (API)|2026-11-07 00:00:00|2026-11-13 23:00:00|
|df_aqicn    |100  |13      |distancia_km, estacion_uid, ronda|instante con zona        |2026-11-09 13:00:00|2026-11-09 21:00:00|
|df_mongo_bpm|100  |7       |fecha desc, id desc              |instante UTC (BSON)      |2026-11-13 23:48:10|2026-11-14 00:16:26|
|df_mysql_dht|100  |5       |fecha desc, id desc              |sin zona declarada       |2026-11-13 18:48:10|2026-11-13 19:16:26|
+------------+-----+--------+---------------------------------+-------------------------+-

## 4. Integración de datos (RA3)

### 4.1 Estrategia de integración

**Diagnóstico (control previo a la integración):** las cuatro fuentes no comparten un período ni una granularidad comunes.

| Fuente | Cobertura | Granularidad | Clave |
|---|---|---|---|
| OpenMeteo | 7 días pasados | 1 hora | `fecha_hora` |
| AQICN | última lectura publicada por ~50 estaciones | 1 hora por estación | (`estacion_uid`, `ronda`) |
| MongoDB (ESP32) | ~30 minutos recientes | ~17 segundos | `id` |
| MySQL (ESP32) | los mismos ~30 minutos, sin zona horaria declarada | ~17 segundos | `id` |

Un join por fecha entre las cuatro fuentes dejaría casi todas las filas sin pareja, y un join por ubicación no aplica porque los sensores no informan coordenadas. Por eso **no se usa una clave temporal común**.

**Decisión:** índice ordinal reproducible por fuente.

1. Cada DataFrame recibe `idx = 1…100` con `row_number()` sobre su **orden documentado**. Cada orden termina en la clave real de la fuente, de modo que no hay empates y el índice es el mismo en cada ejecución.
2. Las columnas se renombran con un prefijo por fuente (`om_`, `aq_`, `mg_`, `my_`), para no repetir nombres (por ejemplo `fecha`, `id` o `humedad`) y para que cada valor siga siendo rastreable a su origen.
3. Las cuatro fuentes se unen por `idx` con `inner join`, y se controla la cardinalidad antes y después de cada unión.
4. Se conservan las fechas y claves originales de cada fuente.

**Límite:** el emparejamiento por índice solo sirve para presentar los datos juntos. Una fila de `df_unificado` **no** afirma que la hora de OpenMeteo, la lectura de AQICN y las de los sensores ocurrieron en el mismo momento. La excepción es el par MongoDB–MySQL: la sección 4.5 comprueba que, con este orden, cada par corresponde al mismo evento del ESP32.

### 4.2 Índice ordinal por fuente

El orden de cada fuente es el mismo del control previo a la integración: OpenMeteo por hora ascendente, AQICN por distancia a Santiago, estación y ronda, y MongoDB y MySQL de la lectura más reciente a la más antigua.

In [15]:
from pyspark.sql import Window

# Orden documentado en el control previo a la integración; cada orden termina en la clave real de la fuente,
# por lo que no hay empates y el índice es reproducible.
ORDEN_FUENTES = {
    "om": (df_openmeteo, [F.col("fecha_hora").asc()]),
    "aq": (df_aqicn,     [F.col("distancia_km").asc(), F.col("estacion_uid").asc(), F.col("ronda").asc()]),
    "mg": (df_mongo_bpm, [F.col("fecha").desc(), F.col("id").desc()]),
    "my": (df_mysql_dht, [F.col("fecha").desc(), F.col("id").desc()]),
}


def indexar(prefijo, df, orden):
    """Agrega idx = 1..n según el orden explícito y antepone el prefijo de la fuente a cada columna."""
    return (df.withColumn("idx", F.row_number().over(Window.orderBy(*orden)))
              .select("idx", *[F.col(c).alias(f"{prefijo}_{c}") for c in df.columns]))


indexados = {p: indexar(p, df, orden) for p, (df, orden) in ORDEN_FUENTES.items()}
for p, df in indexados.items():
    minimo, maximo, distintos, filas = df.agg(F.min("idx"), F.max("idx"), F.countDistinct("idx"), F.count("*")).first()
    assert (minimo, maximo, distintos, filas) == (1, REGISTROS, REGISTROS, REGISTROS), f"{p}: índice no válido"
    print(f"{p}: {filas} filas · idx de {minimo} a {maximo}, sin repetir")

om: 100 filas · idx de 1 a 100, sin repetir


aq: 100 filas · idx de 1 a 100, sin repetir


mg: 100 filas · idx de 1 a 100, sin repetir


my: 100 filas · idx de 1 a 100, sin repetir


### 4.3 `df_unificado` y control de cardinalidad

Cada unión debe mantener exactamente 100 filas: si una fuente tuviera un índice repetido o faltante, la cantidad cambiaría y la celda se detendría.

In [16]:
df_unificado = indexados["om"]
for p in ["aq", "mg", "my"]:
    antes = df_unificado.count()
    df_unificado = df_unificado.join(indexados[p], on="idx", how="inner")
    despues = df_unificado.count()
    print(f"Unión con {p}: {antes} filas antes · {indexados[p].count()} en la fuente · {despues} después")
    assert despues == REGISTROS, f"La unión con {p} cambió la cardinalidad"

df_unificado = df_unificado.orderBy("idx").cache()
columnas_esperadas = 1 + sum(len(df.columns) for df, _ in ORDEN_FUENTES.values())
assert df_unificado.count() == REGISTROS and len(df_unificado.columns) == columnas_esperadas
assert df_unificado.select("idx").distinct().count() == REGISTROS
assert {"mg_altura_metros", "om_condicion_clima"} <= set(df_unificado.columns)
print(f"df_unificado: {df_unificado.count()} filas · {len(df_unificado.columns)} columnas "
      f"(idx + {columnas_esperadas - 1} columnas de las cuatro fuentes)")
df_unificado.printSchema()

Unión con aq: 100 filas antes · 100 en la fuente · 100 después


Unión con mg: 100 filas antes · 100 en la fuente · 100 después


Unión con my: 100 filas antes · 100 en la fuente · 100 después


df_unificado: 100 filas · 36 columnas (idx + 35 columnas de las cuatro fuentes)
root
 |-- idx: integer (nullable = false)
 |-- om_indice_hora: integer (nullable = false)
 |-- om_fecha_hora: timestamp_ntz (nullable = false)
 |-- om_ciudad: string (nullable = false)
 |-- om_temperatura_2m: double (nullable = true)
 |-- om_humedad_relativa_2m: double (nullable = true)
 |-- om_precipitacion: double (nullable = true)
 |-- om_velocidad_viento_10m: double (nullable = true)
 |-- om_direccion_viento_10m: double (nullable = true)
 |-- om_indice_uv: double (nullable = true)
 |-- om_condicion_clima: string (nullable = false)
 |-- aq_estacion_uid: integer (nullable = false)
 |-- aq_estacion: string (nullable = false)
 |-- aq_latitud: double (nullable = false)
 |-- aq_longitud: double (nullable = false)
 |-- aq_distancia_km: double (nullable = false)
 |-- aq_fecha_observacion: timestamp (nullable = false)
 |-- aq_fecha_captura: timestamp (nullable = false)
 |-- aq_ronda: integer (nullable = false)
 

### 4.4 Muestra y cobertura por fuente

La tabla de cobertura confirma que `df_unificado` conserva, para cada fuente, sus 100 claves distintas, sus fechas originales y sus nulos (que no se imputan).

In [17]:
df_unificado.select(
    "idx", "om_fecha_hora", "om_temperatura_2m", "om_condicion_clima",
    "aq_estacion", "aq_pm25", "mg_fecha", F.round("mg_altura_metros", 3).alias("mg_altura_metros"), "my_fecha", "my_temperatura_interior"
).show(5, truncate=False)

COBERTURA = {   # prefijo: (fuente, columna de fecha, clave)
    "om": ("OpenMeteo", "fecha_hora", ["fecha_hora"]),
    "aq": ("AQICN", "fecha_observacion", ["estacion_uid", "ronda"]),
    "mg": ("MongoDB", "fecha", ["id"]),
    "my": ("MySQL", "fecha", ["id"]),
}
filas_cobertura = []
for p, (fuente, col_fecha, clave) in COBERTURA.items():
    mediciones = [c for c in df_unificado.columns if c.startswith(p + "_")]
    fila = df_unificado.agg(
        F.min(f"{p}_{col_fecha}").cast("string").alias("desde"),
        F.max(f"{p}_{col_fecha}").cast("string").alias("hasta"),
        F.countDistinct(*[f"{p}_{c}" for c in clave]).alias("claves_distintas"),
        sum(F.sum(F.col(c).isNull().cast("int")) for c in mediciones).alias("nulos"),
    ).first()
    filas_cobertura.append((fuente, len(mediciones), fila.claves_distintas, fila.nulos, fila.desde[:19], fila.hasta[:19]))
spark.createDataFrame(filas_cobertura, "fuente STRING, columnas INT, claves_distintas BIGINT, nulos BIGINT, "
                                       "desde STRING, hasta STRING").show(truncate=False)

+---+-------------------+-----------------+------------------+-----------------------+-------+-----------------------+----------------+-------------------+-----------------------+
|idx|om_fecha_hora      |om_temperatura_2m|om_condicion_clima|aq_estacion            |aq_pm25|mg_fecha               |mg_altura_metros|my_fecha           |my_temperatura_interior|
+---+-------------------+-----------------+------------------+-----------------------+-------+-----------------------+----------------+-------------------+-----------------------+
|1  |2026-11-07 00:00:00|13.0             |CLIMA MODERADO    |Parque O'Higgins, Chile|25.0   |2026-11-14 00:16:26.706|267.331         |2026-11-13 19:16:26|23.5                   |
|2  |2026-11-07 02:00:00|11.5             |CLIMA MODERADO    |Parque O'Higgins, Chile|25.0   |2026-11-14 00:16:09.592|267.502         |2026-11-13 19:16:09|23.5                   |
|3  |2026-11-07 03:00:00|10.9             |CLIMA MODERADO    |Cerro Navia, Chile     |50.0   |2026-1

+---------+--------+----------------+-----+-------------------+-------------------+
|fuente   |columnas|claves_distintas|nulos|desde              |hasta              |
+---------+--------+----------------+-----+-------------------+-------------------+
|OpenMeteo|10      |100             |0    |2026-11-07 00:00:00|2026-11-13 23:00:00|
|AQICN    |13      |100             |236  |2026-11-09 13:00:00|2026-11-09 21:00:00|
|MongoDB  |7       |100             |0    |2026-11-13 23:48:10|2026-11-14 00:16:26|
|MySQL    |5       |100             |0    |2026-11-13 18:48:10|2026-11-13 19:16:26|
+---------+--------+----------------+-----+-------------------+-------------------+



### 4.5 Comprobación de la estrategia: MongoDB y MySQL

Las fechas de ambos sensores coinciden, pero MySQL las registra 5 horas antes que la hora de Santiago (su zona horaria no está declarada). Si el índice ordinal empareja el mismo evento del ESP32, la diferencia entre `mg_fecha` y `my_fecha` debe estar entre 5 h y 5 h + 2 s en los 100 pares. El margen existe porque MySQL guarda la fecha sin fracción de segundo y las dos inserciones no son simultáneas al milisegundo. Además, los IDs de MySQL deben ser consecutivos (`my_id + idx` constante), es decir, sin lecturas faltantes. Si la comprobación fallara (por ejemplo, porque entró una lectura nueva entre las dos extracciones), basta con volver a ejecutar las secciones 2.3 y 2.4 seguidas.

In [18]:
CINCO_HORAS = 5 * 3600
desfase = df_unificado.select(
    "idx", "mg_id", "mg_fecha", "my_id", "my_fecha",
    F.round(F.col("mg_fecha").cast("double")
            - F.col("my_fecha").cast("timestamp").cast("double"), 3).alias("desfase_s"))
print("Diferencia MongoDB − MySQL en los 100 pares del índice (segundos):")
desfase.agg(F.min("desfase_s").alias("minimo"), F.max("desfase_s").alias("maximo"),
            F.round(F.avg("desfase_s"), 3).alias("promedio")).show()
pares_evento = desfase.filter(F.col("desfase_s").between(CINCO_HORAS, CINCO_HORAS + 2)).count()
ids_consecutivos = df_unificado.select((F.col("my_id") + F.col("idx")).alias("suma")).distinct().count() == 1
print(f"{pares_evento} de {REGISTROS} pares difieren entre 5 h y 5 h + 2 s.")
print("IDs de MySQL consecutivos, sin lecturas faltantes:", ids_consecutivos)
desfase.show(5, truncate=False)

Diferencia MongoDB − MySQL en los 100 pares del índice (segundos):


+---------+---------+---------+
|   minimo|   maximo| promedio|
+---------+---------+---------+
|18000.166|18001.175|18000.649|
+---------+---------+---------+



100 de 100 pares difieren entre 5 h y 5 h + 2 s.
IDs de MySQL consecutivos, sin lecturas faltantes: True
+---+------------------------+-----------------------+------+-------------------+---------+
|idx|mg_id                   |mg_fecha               |my_id |my_fecha           |desfase_s|
+---+------------------------+-----------------------+------+-------------------+---------+
|1  |6af7d28ac1ab6b5957d4c32a|2026-11-14 00:16:26.706|373131|2026-11-13 19:16:26|18000.706|
|2  |6af7d279c1ab6b5957d4c328|2026-11-14 00:16:09.592|373130|2026-11-13 19:16:09|18000.592|
|3  |6af7d268c1ab6b5957d4c326|2026-11-14 00:15:52.638|373129|2026-11-13 19:15:52|18000.638|
|4  |6af7d257c1ab6b5957d4c324|2026-11-14 00:15:35.542|373128|2026-11-13 19:15:35|18000.542|
|5  |6af7d246c1ab6b5957d4c322|2026-11-14 00:15:18.433|373127|2026-11-13 19:15:18|18000.433|
+---+------------------------+-----------------------+------+-------------------+---------+
only showing top 5 rows


**Lectura de la integración (ejecución del 14/11/2026).** Las tres uniones mantuvieron 100 filas, y `df_unificado` quedó con 36 columnas (`idx` y 35 columnas de las cuatro fuentes). La tabla de cobertura confirma 100 claves distintas por fuente. Los 236 nulos son todos de AQICN: corresponden a estaciones que no miden PM10, O3, NO2 o SO2, y no se imputan. La comprobación 4.5 muestra que cada fila de `df_unificado` reúne la misma lectura del ESP32 guardada en MongoDB y en MySQL: en los 100 pares, la diferencia va de 18 000,17 a 18 001,18 s (5 h más una fracción de segundo), y los IDs de MySQL son consecutivos. En cambio, las columnas de OpenMeteo y AQICN solo se presentan junto a ellas.

## 5. Consultas y extracción de datos (ID3.1, ID4.1)

### 5.1 Vistas temporales

Se registra una vista temporal por cada DataFrame **ya transformado** (`openmeteo` incluye `condicion_clima` y `mongo_bpm` incluye `altura_metros`). Además se registra `openmeteo_serie`, la serie horaria completa de 168 horas, que se usa en C3 para contrastar la muestra de 100 horas con la semana completa. Cada consulta se ejecuta en Spark SQL, en su propia celda.

In [19]:
VISTAS = {
    "openmeteo": df_openmeteo,
    "aqicn": df_aqicn,
    "mongo_bpm": df_mongo_bpm,
    "mysql_dht": df_mysql_dht,
    "openmeteo_serie": df_openmeteo_serie,   # apoyo para C3: serie horaria completa de 168 horas
}
for nombre, df in VISTAS.items():
    df.createOrReplaceTempView(nombre)
print("Vistas temporales registradas:", ", ".join(VISTAS))
spark.sql("""
    SELECT 'openmeteo' AS vista, COUNT(*) AS filas FROM openmeteo
    UNION ALL SELECT 'aqicn', COUNT(*) FROM aqicn
    UNION ALL SELECT 'mongo_bpm', COUNT(*) FROM mongo_bpm
    UNION ALL SELECT 'mysql_dht', COUNT(*) FROM mysql_dht
    UNION ALL SELECT 'openmeteo_serie', COUNT(*) FROM openmeteo_serie
""").show()

Vistas temporales registradas: openmeteo, aqicn, mongo_bpm, mysql_dht, openmeteo_serie


+---------------+-----+
|          vista|filas|
+---------------+-----+
|      openmeteo|  100|
|          aqicn|  100|
|      mongo_bpm|  100|
|      mysql_dht|  100|
|openmeteo_serie|  168|
+---------------+-----+



### C1 · OpenMeteo: 10 registros con mayor temperatura (ID4.1)

Orden descendente por temperatura; los empates se resuelven por hora ascendente y los nulos quedan al final.

In [20]:
spark.sql("""
    SELECT fecha_hora, temperatura_2m, humedad_relativa_2m, precipitacion, condicion_clima
    FROM openmeteo
    ORDER BY temperatura_2m DESC NULLS LAST, fecha_hora ASC
    LIMIT 10
""").show(truncate=False)

+-------------------+--------------+-------------------+-------------+---------------+
|fecha_hora         |temperatura_2m|humedad_relativa_2m|precipitacion|condicion_clima|
+-------------------+--------------+-------------------+-------------+---------------+
|2026-11-08 16:00:00|32.2          |16.0               |0.0          |CALOR SECO     |
|2026-11-08 15:00:00|31.6          |15.0               |0.0          |CALOR SECO     |
|2026-11-12 15:00:00|31.3          |33.0               |0.0          |CALOR SECO     |
|2026-11-12 17:00:00|31.3          |36.0               |0.0          |CALOR SECO     |
|2026-11-08 18:00:00|30.5          |26.0               |0.0          |CALOR SECO     |
|2026-11-12 18:00:00|30.5          |39.0               |0.0          |CALOR SECO     |
|2026-11-11 15:00:00|30.2          |35.0               |0.0          |CALOR SECO     |
|2026-11-13 15:00:00|30.2          |37.0               |0.0          |CALOR SECO     |
|2026-11-13 16:00:00|29.9          |38.0   

**Lectura de C1.** La hora más cálida de la semana fue el domingo 08/11 a las 16:00, con **32,2 °C** y 16 % de humedad. Ocho de las diez horas superan los 30 °C y son las ocho horas CALOR SECO de la muestra (08/11, 11/11, 12/11 y 13/11 en la tarde), con humedad de 15 a 39 %. Las otras dos quedan bajo 30 °C (29,9 y 29,7 °C, CLIMA MODERADO). Ninguna hora combinó más de 30 °C con humedad sobre 70 %, por lo que no aparece CALOR Y HUMEDAD.

### C2 · OpenMeteo: humedad relativa promedio de un día y mes (ID4.1)

Como la muestra se renueva en cada ejecución, el día elegido es el **último día presente en la muestra**; así la consulta siempre tiene datos. El filtro se aplica por mes y día, como pide el enunciado, y se informa cuántos valores válidos entran en el promedio.

In [21]:
spark.sql("""
    WITH dia_elegido AS (                       -- último día presente en la muestra
        SELECT MAX(to_date(fecha_hora)) AS fecha FROM openmeteo
    )
    SELECT d.fecha                                   AS fecha_elegida,
           month(d.fecha)                            AS mes,
           day(d.fecha)                              AS dia,
           COUNT(*)                                  AS horas_en_muestra,
           COUNT(o.humedad_relativa_2m)              AS valores_validos,
           ROUND(AVG(o.humedad_relativa_2m), 2)      AS humedad_promedio_pct
    FROM openmeteo o
    JOIN dia_elegido d
      ON month(o.fecha_hora) = month(d.fecha) AND day(o.fecha_hora) = day(d.fecha)
    GROUP BY d.fecha
""").show(truncate=False)

+-------------+---+---+----------------+---------------+--------------------+
|fecha_elegida|mes|dia|horas_en_muestra|valores_validos|humedad_promedio_pct|
+-------------+---+---+----------------+---------------+--------------------+
|2026-11-13   |11 |13 |14              |14             |55.0                |
+-------------+---+---+----------------+---------------+--------------------+



**Lectura de C2.** El 13/11 (mes 11, día 13) la humedad relativa promedio fue **55,0 %**, calculada con las 14 horas de ese día presentes en la muestra, todas con dato. Es el promedio de las horas muestreadas, no de las 24 horas del día.

### C3 · OpenMeteo: horas con precipitación sobre 2 mm en la última semana (ID4.1)

La primera fila responde la consulta sobre `df_openmeteo` (100 horas). La segunda aplica el mismo criterio a la serie completa de los 7 días (168 horas), para distinguir el conteo de la muestra del total real de la semana.

In [22]:
spark.sql("""
    SELECT 'muestra de 100 horas' AS alcance, MAX(ciudad) AS ciudad,
           MIN(fecha_hora) AS desde, MAX(fecha_hora) AS hasta,
           COUNT(DISTINCT to_date(fecha_hora)) AS dias, COUNT(*) AS horas,
           COUNT(precipitacion) AS horas_con_dato,
           COUNT_IF(precipitacion > 2) AS horas_sobre_2mm,
           ROUND(SUM(precipitacion), 1) AS precipitacion_total_mm
    FROM openmeteo
    UNION ALL
    SELECT 'serie completa (7 días)', MAX(ciudad), MIN(fecha_hora), MAX(fecha_hora),
           COUNT(DISTINCT to_date(fecha_hora)), COUNT(*), COUNT(precipitacion),
           COUNT_IF(precipitacion > 2), ROUND(SUM(precipitacion), 1)
    FROM openmeteo_serie
""").show(truncate=False)

+-----------------------+--------+-------------------+-------------------+----+-----+--------------+---------------+----------------------+
|alcance                |ciudad  |desde              |hasta              |dias|horas|horas_con_dato|horas_sobre_2mm|precipitacion_total_mm|
+-----------------------+--------+-------------------+-------------------+----+-----+--------------+---------------+----------------------+
|muestra de 100 horas   |Santiago|2026-11-07 00:00:00|2026-11-13 23:00:00|7   |100  |100           |0              |3.5                   |
|serie completa (7 días)|Santiago|2026-11-07 00:00:00|2026-11-13 23:00:00|7   |168  |168           |0              |5.3                   |
+-----------------------+--------+-------------------+-------------------+----+-----+--------------+---------------+----------------------+



**Lectura de C3.** Entre el 07/11 00:00 y el 13/11 23:00, **ninguna hora superó los 2 mm** de precipitación en Santiago: 0 de 100 horas en la muestra y 0 de 168 en la serie completa. La semana fue seca, con 3,5 mm acumulados en la muestra y 5,3 mm en la serie completa. El cero es un resultado válido, porque la consulta cubre siete días completos y todas las horas tienen dato.

### C4 · AQICN: promedio de PM2.5, PM10, O3, NO2 y SO2 (ID4.1)

Los valores son **sub-índices AQI (escala US EPA)**, no concentraciones en µg/m³. `AVG` ignora los nulos, por eso se informa la cantidad de valores válidos de cada contaminante. Como la segunda ronda de capturas repite lecturas de la primera, el promedio se calcula sobre las 100 capturas y también sobre las observaciones distintas (estación y hora de observación).

In [23]:
spark.sql("""
    WITH observaciones AS (            -- una fila por lectura real (estación, hora de observación)
        SELECT DISTINCT estacion_uid, fecha_observacion, pm25, pm10, o3, no2, so2 FROM aqicn
    ),
    largo AS (
        SELECT '1. 100 capturas' AS alcance,
               stack(5, 'PM2.5', pm25, 'PM10', pm10, 'O3', o3, 'NO2', no2, 'SO2', so2) AS (contaminante, valor)
        FROM aqicn
        UNION ALL
        SELECT '2. observaciones distintas',
               stack(5, 'PM2.5', pm25, 'PM10', pm10, 'O3', o3, 'NO2', no2, 'SO2', so2)
        FROM observaciones
    )
    SELECT contaminante, alcance, 'sub-índice AQI (US EPA)' AS escala,
           COUNT(valor) AS valores_validos, ROUND(AVG(valor), 2) AS promedio
    FROM largo
    GROUP BY contaminante, alcance
    ORDER BY array_position(array('PM2.5', 'PM10', 'O3', 'NO2', 'SO2'), contaminante), alcance
""").show(truncate=False)

+------------+--------------------------+-----------------------+---------------+--------+
|contaminante|alcance                   |escala                 |valores_validos|promedio|
+------------+--------------------------+-----------------------+---------------+--------+
|PM2.5       |1. 100 capturas           |sub-índice AQI (US EPA)|98             |33.61   |
|PM2.5       |2. observaciones distintas|sub-índice AQI (US EPA)|49             |33.61   |
|PM10        |1. 100 capturas           |sub-índice AQI (US EPA)|74             |30.08   |
|PM10        |2. observaciones distintas|sub-índice AQI (US EPA)|37             |30.08   |
|O3          |1. 100 capturas           |sub-índice AQI (US EPA)|32             |13.46   |
|O3          |2. observaciones distintas|sub-índice AQI (US EPA)|16             |13.46   |
|NO2         |1. 100 capturas           |sub-índice AQI (US EPA)|34             |9.46    |
|NO2         |2. observaciones distintas|sub-índice AQI (US EPA)|17             |9.46    |

**Lectura de C4.** En la escala de sub-índices AQI, el promedio más alto fue el de PM2.5 (33,61, con 49 observaciones distintas). Le siguen PM10 (30,08), O3 (13,46), NO2 (9,46) y SO2 (2,85). Todos los promedios están en el tramo 0–50, que la escala US EPA califica como "bueno". Los promedios son iguales con las 100 capturas y con las observaciones distintas, porque la segunda ronda repite exactamente la primera: las capturas repetidas solo duplican los conteos. O3, NO2 y SO2 tienen pocos valores (16, 17 y 13 estaciones), porque la mayoría de las estaciones no los mide. Las lecturas corresponden al 09/11, entre las 13:00 y las 21:00.

### C5 · AQICN: registros con NO2 superior a 4 (ID4.1)

El umbral se aplica en la escala que entrega la fuente (sub-índice AQI). "Superior a 4" es estricto: un valor igual a 4 no cuenta, y la celda lo comprueba antes con casos de frontera.

In [24]:
frontera = spark.sql("SELECT COUNT_IF(no2 > 4) AS n FROM VALUES (3.9), (4.0), (4.1), (NULL) AS t(no2)").first().n
assert frontera == 1, "El filtro debe excluir NO2 = 4"
print("Prueba de frontera: de 3.9, 4.0, 4.1 y NULL solo cuenta 4.1")
spark.sql("""
    SELECT COUNT_IF(no2 > 4)                                       AS capturas_no2_mayor_4,
           COUNT(no2)                                              AS capturas_con_no2,
           COUNT_IF(no2 = 4)                                       AS capturas_no2_igual_4_excluidas,
           COUNT(DISTINCT CASE WHEN no2 > 4 THEN estacion_uid END) AS estaciones_no2_mayor_4,
           'sub-índice AQI (US EPA)'                               AS escala
    FROM aqicn
""").show(truncate=False)

Prueba de frontera: de 3.9, 4.0, 4.1 y NULL solo cuenta 4.1


+--------------------+----------------+------------------------------+----------------------+-----------------------+
|capturas_no2_mayor_4|capturas_con_no2|capturas_no2_igual_4_excluidas|estaciones_no2_mayor_4|escala                 |
+--------------------+----------------+------------------------------+----------------------+-----------------------+
|32                  |34              |0                             |16                    |sub-índice AQI (US EPA)|
+--------------------+----------------+------------------------------+----------------------+-----------------------+



**Lectura de C5.** **32 de las 34 capturas** con NO2 superan 4. Corresponden a 16 de las 17 estaciones que miden NO2. Ninguna captura vale exactamente 4, así que la frontera no altera el resultado, y la prueba previa confirma que un 4 quedaría fuera. El umbral se aplica en la escala AQI, donde 4 es un valor muy bajo: superarlo no indica mala calidad del aire, ya que el promedio de NO2 (9,46) está en el tramo "bueno".

### C6 · MongoDB: primeros 10 registros con `altura_metros` (ID3.1)

Según el orden documentado: lectura más reciente primero (`fecha` descendente, `id` descendente).

In [25]:
spark.sql("""
    SELECT id, fecha, alturaft, ROUND(altura_metros, 3) AS altura_metros,
           presion_atmosferica, monoxido_carbono
    FROM mongo_bpm
    ORDER BY fecha DESC, id DESC
    LIMIT 10
""").show(truncate=False)

+------------------------+-----------------------+--------+-------------+-------------------+----------------+
|id                      |fecha                  |alturaft|altura_metros|presion_atmosferica|monoxido_carbono|
+------------------------+-----------------------+--------+-------------+-------------------+----------------+
|6af7d28ac1ab6b5957d4c32a|2026-11-14 00:16:26.706|877.07  |267.331      |0.968705           |3.62            |
|6af7d279c1ab6b5957d4c328|2026-11-14 00:16:09.592|877.63  |267.502      |0.968685           |3.62            |
|6af7d268c1ab6b5957d4c326|2026-11-14 00:15:52.638|877.35  |267.416      |0.968695           |3.62            |
|6af7d257c1ab6b5957d4c324|2026-11-14 00:15:35.542|877.63  |267.502      |0.968754           |3.62            |
|6af7d246c1ab6b5957d4c322|2026-11-14 00:15:18.433|877.07  |267.331      |0.968675           |3.62            |
|6af7d235c1ab6b5957d4c320|2026-11-14 00:15:01.332|876.79  |267.246      |0.968715           |3.65            |
|

**Lectura de C6.** Las 10 lecturas más recientes del ESP32 (14/11, de 00:13 a 00:16) indican una altura de 876,8 a 879,0 pies, es decir, **267,2 a 267,9 m**, con una presión de 0,9687 atm. Esa altura coincide con la que da la atmósfera estándar para 0,9687 atm (unos 267 m), lo que sugiere que el sensor estima la altura a partir de la presión. La variación de unos ±0,4 m entre lecturas consecutivas es ruido del sensor. El monóxido de carbono se mantiene entre 3,62 y 3,65 (ppm, según el nombre del campo).

### C7 · OpenMeteo: primeros 10 registros con `condicion_clima` (ID3.1)

Según el orden documentado: hora ascendente desde el inicio de la semana.

In [26]:
spark.sql("""
    SELECT fecha_hora, temperatura_2m, humedad_relativa_2m, precipitacion,
           velocidad_viento_10m, direccion_viento_10m, indice_uv, condicion_clima
    FROM openmeteo
    ORDER BY fecha_hora ASC
    LIMIT 10
""").show(truncate=False)

+-------------------+--------------+-------------------+-------------+--------------------+--------------------+---------+---------------+
|fecha_hora         |temperatura_2m|humedad_relativa_2m|precipitacion|velocidad_viento_10m|direccion_viento_10m|indice_uv|condicion_clima|
+-------------------+--------------+-------------------+-------------+--------------------+--------------------+---------+---------------+
|2026-11-07 00:00:00|13.0          |49.0               |0.0          |5.2                 |25.0                |0.0      |CLIMA MODERADO |
|2026-11-07 02:00:00|11.5          |60.0               |0.0          |2.3                 |51.0                |0.0      |CLIMA MODERADO |
|2026-11-07 03:00:00|10.9          |65.0               |0.0          |1.9                 |139.0               |0.0      |CLIMA MODERADO |
|2026-11-07 05:00:00|9.8           |75.0               |0.0          |0.9                 |53.0                |0.0      |CLIMA MODERADO |
|2026-11-07 07:00:00|8.6   

**Lectura de C7.** Las primeras 10 horas de la muestra (07/11, de 00:00 a 15:00) son CLIMA MODERADO, con temperaturas de 8,6 a 24,6 °C. La humedad sube hasta 80 % al amanecer (07:00) y baja a 22 % a las 15:00, a medida que sube la temperatura, y el índice UV llega a 8,7 a las 13:00. En las 100 horas de la muestra solo aparecen dos ramas: CLIMA MODERADO (92 horas) y CALOR SECO (8 horas).

## 6. Fuentes y atribuciones

- Notebooks base del docente (Semana 3, MCDI502): `-BaseEjemploOpenMeteo.ipynb`, `-BaseEjemploAQICN.ipynb`, `-BaseEjemploMongoDB.ipynb` y `-BaseEjemploMysqlPHP_APIRest.ipynb`.
- OpenMeteo, Weather Forecast API: https://open-meteo.com/en/docs
- World Air Quality Index Project, JSON API: https://aqicn.org/json-api/doc/ (datos usados con fines académicos, con atribución a la red de monitoreo de cada estación).
- PyMongo, consultas con orden y límite: https://www.mongodb.com/docs/languages/python/pymongo-driver/current/crud/query/specify-documents-to-return/
- Apache Spark 4, PySpark API: https://spark.apache.org/docs/latest/api/python/index.html